# Preparación, limpieza y estructuración de datos · Vehículos eléctricos y PM2.5

**Actividad:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas
**Proyecto:** Vehículos eléctricos y calidad del aire: ¿el aumento de carros eléctricos ayuda a reducir el PM2.5 en las ciudades?
**Aprendiz:** Juan Camilo Sarrazola Agudelo · Ficha 3169901 · SENA

**Pregunta analítica (EV02):** ¿El aumento de vehículos eléctricos en las ciudades se relaciona con una baja del PM2.5 (µg/m³) entre 2021 y 2025?

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos del proyecto para obtener información confiable que
permita responder la pregunta analítica?

| Sección | Lección del material | Resultado |
|---|---|---|
| 0. Entorno y estado inicial | — | Datos originales cargados y perfil de calidad *antes* |
| 1. Limpieza y estandarización | Lección 2 | Cuatro datasets limpios, validados con `assert` y con bitácora de cambios |
| 2. Filtrado, ordenamiento y agrupación | Lección 3 | Tablas de análisis, tablas dinámicas e indicadores |
| 3. Combinación y estructuración | Lección 4 | Tablas consolidadas validadas y exportadas a `data/processed/` |
| 4. Calidad antes y después | — | Comparación del diagnóstico (EV05) con el resultado final |

> Este notebook parte del diagnóstico de `notebooks/data_quality.ipynb` (EV05). Los archivos de `data/raw/` **no se
> modifican**: solo se leen. Todo lo que se transforma se guarda en `data/processed/`. Las cifras de las
> interpretaciones se calculan con Pandas, no se escriben a mano.

---
## 0. Preparación del entorno y estado inicial de los datos

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)


def md(texto):
    """Muestra texto con formato Markdown dentro del notebook."""
    display(Markdown(texto))


def fmt(n, dec=0):
    """Número con punto de miles y coma decimal: 254371.5 -> 254.371,5"""
    return f"{n:,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(x, dec=2):
    """Porcentaje con coma decimal: 12.345 -> 12,35 %"""
    return fmt(x, dec) + " %"


# Funciona igual si el notebook se abre desde notebooks/ o desde la raíz del proyecto
RAIZ = Path("..") if Path("../data/raw").exists() else Path(".")
RUTA_RAW = RAIZ / "data" / "raw"
RUTA_PROCESSED = RAIZ / "data" / "processed"
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)
print("Datos originales:", RUTA_RAW.as_posix(), "| Datos procesados:", RUTA_PROCESSED.as_posix())

Python 3.13.18 | pandas 3.0.5 | numpy 2.5.3
Datos originales: ../data/raw | Datos procesados: ../data/processed


### 0.1 Carga de los datasets originales
Se cargan los mismos archivos evaluados en la EV05. El CSV del Banco Mundial trae 4 líneas de encabezado
(`skiprows=4`) y su nombre cambia con cada actualización, por eso se busca por patrón.

In [2]:
def ruta(patron):
    """Busca un archivo de data/raw por nombre o patrón."""
    encontrados = sorted(RUTA_RAW.glob(patron))
    if not encontrados:
        raise FileNotFoundError(f"No se encontró '{patron}' en {RUTA_RAW}. Copie los CSV en data/raw (ver README).")
    return encontrados[0]


raw_registro = pd.read_csv(ruta("Electric_Vehicle_Population_Data.csv"), low_memory=False)
raw_historial = pd.read_csv(ruta("Electric_Vehicle_Population_Size_History_By_County.csv"))
raw_ventas = pd.read_csv(ruta("electric-car-sales.csv"))
raw_pm25 = pd.read_csv(ruta("API_EN.ATM.PM25*.csv"), skiprows=4)
raw_pm25_meta = pd.read_csv(ruta("Metadata_Country_API_EN.ATM.PM25*.csv"))

originales = {"ev_registro": raw_registro, "ev_historial": raw_historial, "ev_ventas": raw_ventas, "pm25": raw_pm25}
for nombre, df in originales.items():
    print(f"{nombre:<13} {df.shape[0]:>9,} filas × {df.shape[1]:>3} columnas".replace(",", "."))

ev_registro     300.142 filas ×  16 columnas
ev_historial     36.913 filas ×  10 columnas
ev_ventas           823 filas ×   4 columnas
pm25                266 filas ×  71 columnas


### 0.2 Perfil de calidad inicial
`perfil_calidad()` resume cada DataFrame en una fila. Se usa ahora (estado inicial) y al final (estado final),
para comparar con las mismas medidas.

In [3]:
def perfil_calidad(df, nombre):
    """Resume la calidad de un DataFrame: tamaño, duplicados, faltantes y tipos de datos."""
    celdas = df.size
    faltantes = int(df.isna().sum().sum())
    return {
        "dataset": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "filas_duplicadas": int(df.duplicated().sum()),
        "celdas_faltantes": faltantes,
        "%_celdas_faltantes": round(faltantes / celdas * 100, 2) if celdas else 0.0,
        "col_texto": df.select_dtypes(include=["object", "string"]).shape[1],
        "col_numericas": df.select_dtypes("number").shape[1],
        "col_fecha": df.select_dtypes("datetime").shape[1],
        "col_categoria": df.select_dtypes("category").shape[1],
    }


calidad_inicial = pd.DataFrame([perfil_calidad(df, n) for n, df in originales.items()]).set_index("dataset")
calidad_inicial

,filas,columnas,filas_duplicadas,celdas_faltantes,%_celdas_faltantes,col_texto,col_numericas,col_fecha,col_categoria
dataset,,,,,,,,,
ev_registro,300142,16,0,924,0.02,10,6,0,0
ev_historial,36913,10,0,234,0.06,4,6,0,0
ev_ventas,823,4,0,16,0.49,2,2,0,0
pm25,266,71,0,9390,49.72,4,67,0,0


### 0.3 Problemas del diagnóstico (EV05) medidos sobre los datos originales
Se vuelven a contar los problemas identificados en el informe de calidad. La misma tabla se recalcula al final
sobre los datos procesados (sección 4).

In [4]:
anios_pm25 = [c for c in raw_pm25.columns if c.isdigit()]
paises_wb = raw_pm25_meta.loc[raw_pm25_meta["Region"].notna(), "Country Code"]

problemas_iniciales = pd.DataFrame(
    [
        ("ev_registro", "Vehículos sin condado (sin ubicación)", raw_registro["County"].isna().sum()),
        ("ev_registro", "Registros con State distinto de WA", raw_registro["State"].ne("WA").sum()),
        ("ev_registro", "Electric Range = 0 (sin dato)", raw_registro["Electric Range"].eq(0).sum()),
        ("ev_registro", "Códigos guardados como número", sum(raw_registro[c].dtype.kind == "f" for c in ["Postal Code", "Legislative District", "2020 GEOID"])),
        ("ev_registro", "Ciudades con mayúsculas inconsistentes", raw_registro["City"].nunique() - raw_registro["City"].str.lower().nunique()),
        ("ev_historial", "Fecha guardada como texto", int(not pd.api.types.is_datetime64_any_dtype(raw_historial["Date"]))),
        ("ev_historial", "Filas sin condado ni estado", raw_historial["County"].isna().sum()),
        ("ev_historial", "Filas de condados fuera de WA", (raw_historial["State"].notna() & raw_historial["State"].ne("WA")).sum()),
        ("ev_ventas", "Entidades que no son países (agregados)", raw_ventas.loc[~raw_ventas["Code"].fillna("").str.fullmatch(r"[A-Z]{3}"), "Entity"].nunique()),
        ("ev_ventas", "Ventas negativas", raw_ventas["Electric cars sold"].lt(0).sum()),
        ("pm25", "Columnas sin nombre (Unnamed)", raw_pm25.columns.str.startswith("Unnamed").sum()),
        ("pm25", "Columnas de año vacías", raw_pm25[anios_pm25].isna().all().sum()),
        ("pm25", "Filas que son agregados regionales", (~raw_pm25["Country Code"].isin(paises_wb)).sum()),
        ("pm25", "Formato ancho (años como columnas)", len(anios_pm25)),
    ],
    columns=["dataset", "problema", "casos_antes"],
)
problemas_iniciales["casos_antes"] = problemas_iniciales["casos_antes"].astype(int)
problemas_iniciales

,dataset,problema,casos_antes
0,ev_registro,Vehículos sin condado (sin ubicación),15
1,ev_registro,Registros con State distinto de WA,789
2,ev_registro,Electric Range = 0 (sin dato),197085
3,ev_registro,Códigos guardados como número,3
4,ev_registro,Ciudades con mayúsculas inconsistentes,2
5,ev_historial,Fecha guardada como texto,1
6,ev_historial,Filas sin condado ni estado,117
7,ev_historial,Filas de condados fuera de WA,27670
8,ev_ventas,Entidades que no son países (agregados),4
9,ev_ventas,Ventas negativas,4


---
## 1. Limpieza y estandarización de datos (Lección 2)

Estrategia:
1. **Funciones reutilizables** (`renombrar`, `quitar_duplicados`, `filtrar`, `estandarizar_texto`, `codigo_a_texto`,
   `cero_a_nan`, `convertir_fecha`, `a_categoria`). Cada una devuelve un DataFrame nuevo (no modifica el original)
   y anota lo que hizo en una **bitácora**.
2. Un **pipeline por dataset** que encadena esas funciones con `pipe()`.
3. **Validaciones** con `assert` al final de cada pipeline. Si una regla no se cumple, el notebook se detiene.

Los nombres de columna se pasan a `snake_case` en inglés, igual que los nombres de archivos y carpetas del proyecto.

### 1.1 Funciones reutilizables de limpieza

In [5]:
bitacora = []


def registrar(dataset, paso, filas_antes, filas_despues, detalle=""):
    """Guarda en la bitácora qué se hizo, a qué dataset y cuántas filas cambiaron."""
    bitacora.append({
        "dataset": dataset,
        "paso": paso,
        "filas_antes": filas_antes,
        "filas_despues": filas_despues,
        "filas_eliminadas": filas_antes - filas_despues,
        "detalle": detalle,
    })


def renombrar(df, mapa, dataset):
    """Renombra columnas según `mapa` y comprueba que todas existan."""
    faltan = set(mapa) - set(df.columns)
    assert not faltan, f"{dataset}: no existen las columnas {faltan}"
    salida = df.rename(columns=mapa)
    registrar(dataset, "Renombrar columnas", len(df), len(salida), f"{len(mapa)} columnas a snake_case en inglés")
    return salida


def eliminar_columnas(df, columnas, dataset, motivo):
    """Quita columnas que no aportan al análisis."""
    salida = df.drop(columns=columnas)
    registrar(dataset, "Eliminar columnas", len(df), len(salida), f"{', '.join(map(str, columnas))}: {motivo}")
    return salida


def quitar_duplicados(df, dataset, subset=None):
    """Elimina filas repetidas (completas o según columnas clave) y conserva la primera."""
    salida = df.drop_duplicates(subset=subset, keep="first")
    criterio = "fila completa" if subset is None else "clave " + ", ".join(subset)
    registrar(dataset, "Eliminar duplicados", len(df), len(salida), criterio)
    return salida


def filtrar(df, condicion, dataset, paso, detalle=""):
    """Conserva las filas que cumplen `condicion` (función que recibe el DataFrame y devuelve una máscara booleana)."""
    salida = df.loc[condicion(df)].copy()
    registrar(dataset, paso, len(df), len(salida), detalle)
    return salida


def estandarizar_texto(df, columnas, dataset, caso=None):
    """Quita espacios sobrantes y unifica mayúsculas: caso = 'title', 'upper' o None."""
    salida = df.copy()
    for col in columnas:
        s = salida[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
        if caso == "title":
            s = s.str.title()
        elif caso == "upper":
            s = s.str.upper()
        salida[col] = s
    registrar(dataset, "Estandarizar texto", len(df), len(salida), f"{', '.join(columnas)} (strip, espacios{', ' + caso if caso else ''})")
    return salida


def codigo_a_texto(df, anchos, dataset):
    """Convierte códigos guardados como número (98177.0, 5.30e+10) a texto de ancho fijo con ceros a la izquierda."""
    salida = df.copy()
    for col, ancho in anchos.items():
        salida[col] = salida[col].astype("Int64").astype("string").str.zfill(ancho)
    registrar(dataset, "Códigos numéricos → texto", len(df), len(salida), ", ".join(f"{c} ({a} dígitos)" for c, a in anchos.items()))
    return salida


def cero_a_nan(df, columna, dataset, motivo):
    """Reemplaza por NaN los ceros que en realidad significan 'sin dato'."""
    salida = df.copy()
    n = int(salida[columna].eq(0).sum())
    salida[columna] = salida[columna].mask(salida[columna].eq(0))
    registrar(dataset, f"0 → NaN en {columna}", len(df), len(salida), f"{fmt(n)} valores; {motivo}")
    return salida


def convertir_fecha(df, columna, formato, dataset):
    """Convierte texto a fecha con un formato explícito; falla si algún valor no se puede convertir."""
    salida = df.copy()
    salida[columna] = pd.to_datetime(salida[columna], format=formato, errors="coerce")
    no_validas = int(salida[columna].isna().sum() - df[columna].isna().sum())
    assert no_validas == 0, f"{dataset}: {no_validas} fechas no se pudieron convertir"
    registrar(dataset, "Texto → fecha", len(df), len(salida), f"{columna} con formato '{formato}'")
    return salida


def a_categoria(df, columnas, dataset):
    """Convierte columnas de pocos valores repetidos al tipo category (menos memoria, orden controlado)."""
    salida = df.copy()
    for col in columnas:
        salida[col] = salida[col].astype("category")
    registrar(dataset, "Texto → category", len(df), len(salida), ", ".join(columnas))
    return salida


def resumen_faltantes(df, decisiones=None):
    """Tabla de columnas con faltantes y, si se indica, la decisión tomada para cada una."""
    n = df.isna().sum()
    tabla = pd.DataFrame({"faltantes": n, "%": (n / len(df) * 100).round(3)}).query("faltantes > 0")
    if decisiones is not None:
        tabla["tratamiento"] = tabla.index.map(lambda c: decisiones.get(c, "—"))
    return tabla

### 1.2 `ev_registro` · registro actual de vehículos eléctricos de Washington

**Evidencia previa a la limpieza.** En la EV05 se vio que 65,66 % de las filas tienen `Electric Range = 0`.
Antes de decidir qué hacer, se cruza con la columna de elegibilidad CAFV para saber si ese 0 es una autonomía real:

In [6]:
pd.crosstab(
    raw_registro["Electric Range"].eq(0).map({True: "Electric Range = 0", False: "Electric Range > 0 o NaN"}),
    raw_registro["Clean Alternative Fuel Vehicle (CAFV) Eligibility"],
)

Clean Alternative Fuel Vehicle (CAFV) Eligibility,Clean Alternative Fuel Vehicle Eligible,Eligibility unknown as battery range has not been researched,Not eligible due to low battery range
Electric Range,,,
Electric Range = 0,0,197085,0
Electric Range > 0 o NaN,78933,0,24124


In [7]:
ceros = raw_registro["Electric Range"].eq(0)
sin_investigar = raw_registro["Clean Alternative Fuel Vehicle (CAFV) Eligibility"].str.startswith("Eligibility unknown")
ciudades_caso = (raw_registro.assign(c=raw_registro["City"].str.lower())
                 .groupby("c")["City"].unique().loc[lambda s: s.str.len() > 1])
md(f"""**Lectura:** los {fmt(ceros.sum())} ceros coinciden {'exactamente' if ceros.equals(sin_investigar) else 'parcialmente'} con
*"Eligibility unknown as battery range has not been researched"*. El 0 significa **"autonomía no investigada"**, no
un vehículo sin autonomía. Si se deja como número, el promedio de autonomía queda subestimado; por eso se reemplaza por `NaN`.

También aparecen ciudades escritas de dos formas: {'; '.join(' / '.join(v) for v in ciudades_caso)}.
Se unifican con `str.title()`.""")

**Lectura:** los 197.085 ceros coinciden exactamente con
*"Eligibility unknown as battery range has not been researched"*. El 0 significa **"autonomía no investigada"**, no
un vehículo sin autonomía. Si se deja como número, el promedio de autonomía queda subestimado; por eso se reemplaza por `NaN`.

También aparecen ciudades escritas de dos formas: McChord AFB / Mcchord Afb; Mccleary / McCleary.
Se unifican con `str.title()`.

**Pipeline de limpieza.** Decisiones y su justificación:

| Paso | Decisión | Justificación |
|---|---|---|
| Duplicados | Eliminar filas completas repetidas y verificar que `dol_vehicle_id` sea único | Un vehículo contado dos veces infla los conteos por condado |
| Sin condado | Excluir | Son direcciones militares o de Canadá; no se pueden asignar a un condado de Washington |
| `state != 'WA'` | Excluir | La pregunta se analiza para los condados de Washington |
| Textos | `strip` + espacios; `county`, `city` en tipo título; `make`, `model` en mayúscula | Un mismo valor escrito distinto se cuenta como dos categorías |
| Códigos | `postal_code`, `legislative_district`, `census_tract_2020` a texto con ceros a la izquierda | Son identificadores: no se suman y como `float` muestran `12.0` o notación científica |
| `electric_range = 0` | Reemplazar por `NaN` | Es "no investigado" (ver cruce anterior) |
| `vehicle_location` | Extraer `longitude` y `latitude` numéricas | El texto `POINT (lon lat)` no se puede usar en mapas ni cálculos |
| Categorías | `ev_type` → BEV/PHEV y elegibilidad CAFV abreviada, como `category` | Textos largos y repetidos; facilitan agrupar |

In [8]:
MAPA_REGISTRO = {
    "VIN (1-10)": "vin_prefix",
    "County": "county",
    "City": "city",
    "State": "state",
    "Postal Code": "postal_code",
    "Model Year": "model_year",
    "Make": "make",
    "Model": "model",
    "Electric Vehicle Type": "ev_type",
    "Clean Alternative Fuel Vehicle (CAFV) Eligibility": "cafv_eligibility",
    "Electric Range": "electric_range",
    "Legislative District": "legislative_district",
    "DOL Vehicle ID": "dol_vehicle_id",
    "Vehicle Location": "vehicle_location",
    "Electric Utility": "electric_utility",
    "2020 GEOID": "census_tract_2020",
}
TIPO_EV = {"Battery Electric Vehicle (BEV)": "BEV", "Plug-in Hybrid Electric Vehicle (PHEV)": "PHEV"}
CAFV = {
    "Clean Alternative Fuel Vehicle Eligible": "Eligible",
    "Not eligible due to low battery range": "Not eligible",
    "Eligibility unknown as battery range has not been researched": "Unknown",
}


def extraer_coordenadas(df, dataset="ev_registro"):
    """Separa 'POINT (lon lat)' en dos columnas numéricas y elimina la columna de texto."""
    coords = df["vehicle_location"].str.extract(r"POINT \((?P<longitude>-?\d+\.?\d*) (?P<latitude>-?\d+\.?\d*)\)").astype("float64")
    salida = df.drop(columns="vehicle_location").join(coords)
    registrar(dataset, "Extraer coordenadas", len(df), len(salida), "vehicle_location → longitude, latitude")
    return salida


def recodificar_tipos(df, dataset="ev_registro"):
    """Abrevia tipo de vehículo y elegibilidad CAFV; falla si aparece un valor no previsto."""
    salida = df.copy()
    salida["ev_type"] = salida["ev_type"].map(TIPO_EV)
    salida["cafv_eligibility"] = salida["cafv_eligibility"].map(CAFV)
    assert salida[["ev_type", "cafv_eligibility"]].notna().all().all(), "Hay valores sin recodificar"
    registrar(dataset, "Recodificar categorías", len(df), len(salida), "ev_type → BEV/PHEV; cafv_eligibility → Eligible/Not eligible/Unknown")
    return salida


def limpiar_registro(df):
    d = "ev_registro"
    return (
        df.pipe(renombrar, MAPA_REGISTRO, d)
        .pipe(quitar_duplicados, d)
        .pipe(filtrar, lambda x: x["county"].notna(), d, "Excluir sin condado", "direcciones militares o fuera de EE. UU.")
        .pipe(filtrar, lambda x: x["state"].eq("WA"), d, "Conservar solo WA", "condados de Washington")
        .pipe(estandarizar_texto, ["county", "city"], d, caso="title")
        .pipe(estandarizar_texto, ["make", "model", "electric_utility", "vin_prefix", "state"], d, caso="upper")
        .pipe(codigo_a_texto, {"postal_code": 5, "legislative_district": 2, "census_tract_2020": 11}, d)
        .pipe(cero_a_nan, "electric_range", d, "autonomía no investigada (CAFV = Unknown)")
        .pipe(extraer_coordenadas)
        .pipe(recodificar_tipos)
        .pipe(a_categoria, ["ev_type", "cafv_eligibility", "county"], d)
        .reset_index(drop=True)
    )


registro = limpiar_registro(raw_registro)
print(f"Filas: {fmt(len(raw_registro))} → {fmt(len(registro))}")
registro.head()

Filas: 300.142 → 299.353


,vin_prefix,county,city,state,postal_code,model_year,make,model,ev_type,cafv_eligibility,electric_range,legislative_district,dol_vehicle_id,electric_utility,census_tract_2020,longitude,latitude
0,3FMTK3SU8P,King,Seattle,WA,98177,2023,FORD,MUSTANG MACH-E,BEV,Unknown,NaN,32,254622924,CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),53033000402,-122.37238,47.74068
1,JM3KKEHA8S,Thurston,Yelm,WA,98597,2025,MAZDA,CX-90,PHEV,Not eligible,26.0,02,277791643,PUGET SOUND ENERGY INC,53067012412,-122.60735,46.94239
2,5YJ3E1EB8J,King,Sammamish,WA,98075,2018,TESLA,MODEL 3,BEV,Eligible,215.0,41,347418676,PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),53033032224,-122.03539,47.61344
3,JHMZC5F12K,Island,Oak Harbor,WA,98277,2019,HONDA,CLARITY,PHEV,Eligible,47.0,10,223620701,PUGET SOUND ENERGY INC,53029970800,-122.64682,48.29077
4,1C4RJXN65R,Kitsap,Bremerton,WA,98312,2024,JEEP,WRANGLER,PHEV,Not eligible,21.0,26,256402674,PUGET SOUND ENERGY INC,53035080600,-122.65223,47.57192


In [9]:
DECISIONES_REGISTRO = {
    "electric_range": "Se deja NaN: 0 = no investigado; no se imputa para no inventar autonomías",
    "legislative_district": "Se deja <NA>: no se usa en el análisis por condado",
    "longitude": "Se deja NaN: el análisis usa county; no afecta conteos",
    "latitude": "Se deja NaN: el análisis usa county; no afecta conteos",
}
resumen_faltantes(registro, DECISIONES_REGISTRO)

,faltantes,%,tratamiento
electric_range,196629,65.685,Se deja NaN: 0 = no investigado; no se imputa ...
legislative_district,1,0.000,Se deja <NA>: no se usa en el análisis por con...
longitude,9,0.003,Se deja NaN: el análisis usa county; no afecta...
latitude,9,0.003,Se deja NaN: el análisis usa county; no afecta...


In [10]:
# Validaciones de ev_registro
assert registro["dol_vehicle_id"].is_unique, "Hay vehículos repetidos"
assert registro.duplicated().sum() == 0
assert registro["state"].eq("WA").all()
assert registro["county"].notna().all() and registro["county"].nunique() == 39, "Washington tiene 39 condados"
assert registro["city"].nunique() == registro["city"].str.lower().nunique(), "Persisten ciudades con distinta escritura"
assert (registro["electric_range"].dropna() > 0).all(), "Quedan autonomías en 0"
assert registro["model_year"].between(1990, pd.Timestamp.today().year + 2).all()
assert registro["ev_type"].isin(["BEV", "PHEV"]).all()
assert registro["postal_code"].str.fullmatch(r"\d{5}").all()
assert registro["census_tract_2020"].str.fullmatch(r"53\d{9}").all(), "Todos los tractos deben ser de WA (FIPS 53)"
assert registro["latitude"].dropna().between(45.5, 49.1).all() and registro["longitude"].dropna().between(-124.9, -116.9).all()
assert len(registro) == raw_registro["State"].eq("WA").sum(), "Solo debían salir los registros fuera de WA"
print("✔ ev_registro: 12 validaciones superadas")
registro.dtypes.to_frame("dtype").T

✔ ev_registro: 12 validaciones superadas


,vin_prefix,county,city,state,postal_code,model_year,make,model,ev_type,cafv_eligibility,electric_range,legislative_district,dol_vehicle_id,electric_utility,census_tract_2020,longitude,latitude
dtype,string,category,string,string,string,int64,string,string,category,category,float64,string,int64,string,string,float64,float64


### 1.3 `ev_historial` · historial mensual por condado

| Paso | Decisión | Justificación |
|---|---|---|
| `Date` | Texto → `datetime64` con formato `'%B %d %Y'` | Sin fecha no se puede ordenar ni agrupar por año |
| Filas sin condado/estado | Excluir | No se pueden asignar a un condado |
| `state != 'WA'` | Excluir | Son condados de otros estados donde viven dueños de vehículos registrados en WA |
| Duplicados | Verificar clave (`date`, `county`, `primary_use`) | Cada combinación debe aparecer una sola vez |
| Nuevas columnas | `year`, `month` | Facilitan filtrar y agrupar por periodo |

In [11]:
MAPA_HISTORIAL = {
    "Date": "date",
    "County": "county",
    "State": "state",
    "Vehicle Primary Use": "primary_use",
    "Battery Electric Vehicles (BEVs)": "bev",
    "Plug-In Hybrid Electric Vehicles (PHEVs)": "phev",
    "Electric Vehicle (EV) Total": "ev_total",
    "Non-Electric Vehicle Total": "non_ev_total",
    "Total Vehicles": "total_vehicles",
    "Percent Electric Vehicles": "pct_ev",
}


def agregar_periodo(df, dataset="ev_historial"):
    """Crea year y month a partir de la fecha."""
    salida = df.assign(year=df["date"].dt.year, month=df["date"].dt.month)
    registrar(dataset, "Columnas derivadas", len(df), len(salida), "year, month desde date")
    return salida


def limpiar_historial(df):
    d = "ev_historial"
    return (
        df.pipe(renombrar, MAPA_HISTORIAL, d)
        .pipe(convertir_fecha, "date", "%B %d %Y", d)
        .pipe(filtrar, lambda x: x["county"].notna() & x["state"].notna(), d, "Excluir sin condado/estado", "no asignables")
        .pipe(filtrar, lambda x: x["state"].eq("WA"), d, "Conservar solo WA", "condados de Washington")
        .pipe(estandarizar_texto, ["county", "state", "primary_use"], d)
        .pipe(quitar_duplicados, d, subset=["date", "county", "primary_use"])
        .pipe(agregar_periodo)
        .pipe(a_categoria, ["primary_use"], d)
        .sort_values(["date", "county", "primary_use"])
        .reset_index(drop=True)
    )


historial = limpiar_historial(raw_historial)
print(f"Filas: {fmt(len(raw_historial))} → {fmt(len(historial))} | periodo {historial['date'].min():%Y-%m} a {historial['date'].max():%Y-%m}")
historial.head()

Filas: 36.913 → 9.126 | periodo 2017-01 a 2026-09


,date,county,state,primary_use,bev,phev,ev_total,non_ev_total,total_vehicles,pct_ev,year,month
0,2017-01-31,Adams,WA,Passenger,1,1,2,12167,12169,0.02,2017,1
1,2017-01-31,Adams,WA,Truck,0,0,0,5077,5077,0.00,2017,1
2,2017-01-31,Asotin,WA,Passenger,4,3,7,12313,12320,0.06,2017,1
3,2017-01-31,Asotin,WA,Truck,0,0,0,6477,6477,0.00,2017,1
4,2017-01-31,Benton,WA,Passenger,129,129,258,127780,128038,0.20,2017,1


In [12]:
# Validaciones de ev_historial
assert pd.api.types.is_datetime64_any_dtype(historial["date"])
assert historial.isna().sum().sum() == 0, "No deben quedar faltantes"
assert historial["state"].eq("WA").all() and historial["county"].nunique() == 39
assert not historial.duplicated(["date", "county", "primary_use"]).any()
assert (historial["bev"] + historial["phev"]).eq(historial["ev_total"]).all(), "BEV + PHEV ≠ total eléctrico"
assert (historial["ev_total"] + historial["non_ev_total"]).eq(historial["total_vehicles"]).all()
assert (historial["ev_total"] / historial["total_vehicles"] * 100 - historial["pct_ev"]).abs().max() < 0.01
assert set(historial["primary_use"].unique()) == {"Passenger", "Truck"}
# cada mes debe tener los 39 condados × 2 usos (serie completa, sin huecos)
filas_por_mes = historial.groupby("date").size()
assert filas_por_mes.eq(39 * 2).all(), f"Meses incompletos: {filas_por_mes[filas_por_mes != 78].to_dict()}"
print(f"✔ ev_historial: 9 validaciones superadas · {historial['date'].nunique()} meses completos")

✔ ev_historial: 9 validaciones superadas · 117 meses completos


### 1.4 `ev_ventas` · ventas anuales de carros eléctricos por país (IEA)

| Paso | Decisión | Justificación |
|---|---|---|
| Agregados | Conservar solo filas con código ISO de 3 letras | *World*, *Europe*, *EU (27)* y *Rest of World* suman países y causarían doble conteo |
| Ventas negativas | Se eliminan con los agregados (todas son de *Rest of World*); se valida que no quede ninguna | Una venta negativa es un residuo de cálculo, no un dato |
| Nombres | `Electric cars sold` → `ev_sales` | Nombre corto y en `snake_case` |

In [13]:
print("Entidades sin código ISO de país:",
      sorted(raw_ventas.loc[~raw_ventas["Code"].fillna("").str.fullmatch(r"[A-Z]{3}"), "Entity"].unique()))
print("Ventas negativas por entidad:", raw_ventas.loc[raw_ventas["Electric cars sold"] < 0, "Entity"].value_counts().to_dict())


def limpiar_ventas(df):
    d = "ev_ventas"
    return (
        df.pipe(renombrar, {"Entity": "country", "Code": "iso3", "Year": "year", "Electric cars sold": "ev_sales"}, d)
        .pipe(estandarizar_texto, ["country", "iso3"], d)
        .pipe(filtrar, lambda x: x["iso3"].fillna("").str.fullmatch(r"[A-Z]{3}"), d, "Excluir agregados", "World, Europe, EU (27), Rest of World")
        .pipe(quitar_duplicados, d, subset=["iso3", "year"])
        .astype({"year": "int64", "ev_sales": "int64"})
        .sort_values(["iso3", "year"])
        .reset_index(drop=True)
    )


ventas = limpiar_ventas(raw_ventas)

assert ventas["iso3"].str.fullmatch(r"[A-Z]{3}").all()
assert ventas["ev_sales"].ge(0).all(), "Quedan ventas negativas"
assert not ventas.duplicated(["iso3", "year"]).any()
assert ventas.isna().sum().sum() == 0
print(f"✔ ev_ventas: 4 validaciones superadas · {ventas['iso3'].nunique()} países · {ventas['year'].min()}–{ventas['year'].max()}")
ventas.head()

Entidades sin código ISO de país: ['Europe', 'European Union (27)', 'Rest of World', 'World']
Ventas negativas por entidad: {'Rest of World': 4}
✔ ev_ventas: 4 validaciones superadas · 60 países · 2010–2025


,country,iso3,year,ev_sales
0,United Arab Emirates,ARE,2012,200
1,United Arab Emirates,ARE,2013,139
2,United Arab Emirates,ARE,2014,49
3,United Arab Emirates,ARE,2015,85
4,United Arab Emirates,ARE,2016,58


### 1.5 `pm25` · exposición media anual a PM2.5 por país (Banco Mundial)

| Paso | Decisión | Justificación |
|---|---|---|
| `Unnamed: 70` | Eliminar | Columna vacía creada por la coma final del CSV |
| `Indicator Name/Code` | Eliminar tras verificar que tienen un solo valor | No aportan información (todo el archivo es el mismo indicador) |
| Años vacíos | Eliminar columnas sin ningún dato | Ruido |
| Agregados | Unir con `Metadata_Country` (`merge` de búsqueda) y conservar filas con `Region` | Separa países de regiones, ingresos y "World" |
| Formato | `melt()` a formato largo (`iso3`, `year`, `pm25`) | Para unir por año con los otros datasets |
| Faltantes | Eliminar filas sin valor de PM2.5 tras el `melt` | En formato largo, un faltante es un país-año que no se midió; imputarlo inventaría contaminación |

In [14]:
assert raw_pm25["Indicator Code"].nunique() == 1 and raw_pm25["Indicator Name"].nunique() == 1


def solo_paises(df, meta, dataset="pm25"):
    """Agrega región y grupo de ingreso desde los metadatos y conserva solo países (Region no vacía)."""
    info = (meta.rename(columns={"Country Code": "iso3", "Region": "region", "IncomeGroup": "income_group"})
            [["iso3", "region", "income_group"]])
    salida = df.merge(info, on="iso3", how="left", validate="one_to_one")
    salida = salida.loc[salida["region"].notna()].copy()
    registrar(dataset, "Conservar solo países", len(df), len(salida), "merge con Metadata_Country y Region no vacía")
    return salida


def a_formato_largo(df, dataset="pm25"):
    """Pasa los años de columnas a filas: una fila por país y año."""
    anios = [c for c in df.columns if str(c).isdigit()]
    salida = df.melt(id_vars=["country", "iso3", "region", "income_group"], value_vars=anios, var_name="year", value_name="pm25")
    salida["year"] = salida["year"].astype("int64")
    registrar(dataset, "Formato ancho → largo (melt)", len(df), len(salida), f"{len(anios)} columnas de año → filas")
    return salida


def limpiar_pm25(df, meta):
    d = "pm25"
    vacias = [c for c in df.columns if c.isdigit() and df[c].isna().all()]
    return (
        df.pipe(eliminar_columnas, [c for c in df.columns if c.startswith("Unnamed")], d, "columna vacía por coma final")
        .pipe(eliminar_columnas, ["Indicator Name", "Indicator Code"], d, "valor constante")
        .pipe(eliminar_columnas, vacias, d, "años sin ningún dato")
        .pipe(renombrar, {"Country Name": "country", "Country Code": "iso3"}, d)
        .pipe(solo_paises, meta)
        .pipe(a_formato_largo)
        .pipe(filtrar, lambda x: x["pm25"].notna(), d, "Excluir país-año sin medición", "faltantes no imputables")
        .pipe(a_categoria, ["region", "income_group"], d)
        .sort_values(["iso3", "year"])
        .reset_index(drop=True)
    )


pm25 = limpiar_pm25(raw_pm25, raw_pm25_meta)

assert not pm25.duplicated(["iso3", "year"]).any()
assert pm25["pm25"].between(0, 200).all(), "PM2.5 fuera de rango plausible"
assert pm25.isna().sum().sum() == 0
assert pm25["iso3"].isin(paises_wb).all()
print(f"✔ pm25: 4 validaciones superadas · {pm25['iso3'].nunique()} países · {pm25['year'].min()}–{pm25['year'].max()}")
pm25.head()

✔ pm25: 4 validaciones superadas · 200 países · 1990–2023


,country,iso3,region,income_group,year,pm25
0,Afghanistan,AFG,Middle East & North Africa,Low income,1990,48.426586
1,Afghanistan,AFG,Middle East & North Africa,Low income,1991,47.941609
2,Afghanistan,AFG,Middle East & North Africa,Low income,1992,47.974404
3,Afghanistan,AFG,Middle East & North Africa,Low income,1993,48.007198
4,Afghanistan,AFG,Middle East & North Africa,Low income,1994,48.039993


### 1.6 Bitácora de cambios y estado inicial vs. final

In [15]:
tabla_bitacora = pd.DataFrame(bitacora)
tabla_bitacora

,dataset,paso,filas_antes,filas_despues,filas_eliminadas,detalle
0,ev_registro,Renombrar columnas,300142,300142,0,16 columnas a snake_case en inglés
1,ev_registro,Eliminar duplicados,300142,300142,0,fila completa
2,ev_registro,Excluir sin condado,300142,300127,15,direcciones militares o fuera de EE. UU.
3,ev_registro,Conservar solo WA,300127,299353,774,condados de Washington
4,ev_registro,Estandarizar texto,299353,299353,0,"county, city (strip, espacios, title)"
5,ev_registro,Estandarizar texto,299353,299353,0,"make, model, electric_utility, vin_prefix, sta..."
6,ev_registro,Códigos numéricos → texto,299353,299353,0,"postal_code (5 dígitos), legislative_district ..."
7,ev_registro,0 → NaN en electric_range,299353,299353,0,196.594 valores; autonomía no investigada (CAF...
8,ev_registro,Extraer coordenadas,299353,299353,0,"vehicle_location → longitude, latitude"
9,ev_registro,Recodificar categorías,299353,299353,0,ev_type → BEV/PHEV; cafv_eligibility → Eligibl...


In [16]:
limpios = {"ev_registro": registro, "ev_historial": historial, "ev_ventas": ventas, "pm25": pm25}
calidad_final = pd.DataFrame([perfil_calidad(df, n) for n, df in limpios.items()]).set_index("dataset")
comparacion = pd.concat({"antes": calidad_inicial, "después": calidad_final}, axis=1).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)
comparacion[["filas", "columnas", "filas_duplicadas", "celdas_faltantes", "%_celdas_faltantes", "col_texto", "col_fecha", "col_categoria"]]

filas         columnas         filas_duplicadas         celdas_faltantes         %_celdas_faltantes         col_texto         col_fecha         col_categoria        
               antes después    antes después            antes después            antes después              antes después     antes después     antes después         antes después
dataset                                                                                                                                                                             
ev_registro   300142  299353       16      17                0       0              924  196648               0.02    3.86        10       9         0       0             0       3
ev_historial   36913    9126       10      12                0       0              234       0               0.06    0.00         4       2         0       1             0       1
ev_ventas        823     759        4       4                0       0               16       0               0.49    0.00         2       2         0       0             0       0
pm25             266    6800       71       6                0       0             9390       0              49.72    0.00         4       2         0       0             0       2

In [17]:
md(f"""**Interpretación.** Los cuatro datasets quedan sin duplicados y con tipos coherentes. Las filas que salen no son
errores borrados al azar: son registros fuera de Washington, agregados regionales o país-años sin medición, cada
uno anotado en la bitácora. En `ev_registro` los faltantes **aumentan** a propósito
({fmt(calidad_inicial.loc['ev_registro', 'celdas_faltantes'])} → {fmt(calidad_final.loc['ev_registro', 'celdas_faltantes'])} celdas),
porque los ceros de autonomía ahora están marcados como `NaN`: es información más honesta, no peor. En `pm25` el número
de filas crece porque el formato largo usa una fila por país y año.""")

**Interpretación.** Los cuatro datasets quedan sin duplicados y con tipos coherentes. Las filas que salen no son
errores borrados al azar: son registros fuera de Washington, agregados regionales o país-años sin medición, cada
uno anotado en la bitácora. En `ev_registro` los faltantes **aumentan** a propósito
(924 → 196.648 celdas),
porque los ceros de autonomía ahora están marcados como `NaN`: es información más honesta, no peor. En `pm25` el número
de filas crece porque el formato largo usa una fila por país y año.

### 1.7 Exportación de los datasets limpios
El registro se guarda comprimido (`.csv.gz`) porque supera los 40 MB sin comprimir; Pandas lo lee igual con `read_csv`.

In [18]:
def exportar(df, nombre):
    """Guarda en data/processed y verifica que el archivo se lea con el mismo número de filas y columnas."""
    destino = RUTA_PROCESSED / nombre
    df.to_csv(destino, index=False)
    releido = pd.read_csv(destino)
    assert releido.shape == df.shape, f"{nombre}: {releido.shape} ≠ {df.shape}"
    return {"archivo": f"data/processed/{nombre}", "filas": len(df), "columnas": df.shape[1], "KB": round(destino.stat().st_size / 1024)}


exportados = [
    exportar(registro, "ev_registry_clean.csv.gz"),
    exportar(historial, "ev_history_wa_clean.csv"),
    exportar(ventas, "ev_sales_clean.csv"),
    exportar(pm25, "pm25_clean.csv"),
]
pd.DataFrame(exportados)

,archivo,filas,columnas,KB
0,data/processed/ev_registry_clean.csv.gz,299353,17,8624
1,data/processed/ev_history_wa_clean.csv,9126,12,560
2,data/processed/ev_sales_clean.csv,759,4,17
3,data/processed/pm25_clean.csv,6800,6,492


---
## 2. Filtrado, ordenamiento y agrupación (Lección 3)

La pregunta tiene dos partes: **¿aumentaron los vehículos eléctricos?** y **¿bajó el PM2.5?** en 2021–2025.
Los indicadores se calculan sobre esa ventana:

| # | Indicador | Dataset | Responde |
|---|---|---|---|
| I1 | Participación de eléctricos en el parque de pasajeros de Washington (% y puntos porcentuales) | `ev_historial` | ¿Cuánto creció la adopción? |
| I2 | Crecimiento de las ventas de eléctricos por país 2021 → 2023 | `ev_ventas` | ¿Dónde creció más la adopción? |
| I3 | Variación del PM2.5 por país 2021 → 2023 (µg/m³ y %) | `pm25` | ¿Bajó la contaminación? |
| I4 | Participación de eléctricos de batería (BEV) entre los modelos 2021–2025 | `ev_registro` | ¿Qué tipo de eléctrico crece? (solo BEV elimina las emisiones del tubo de escape) |

> El PM2.5 del Banco Mundial llega hasta **2023**; por eso los indicadores internacionales usan 2021 → 2023.

### 2.1 Filtros
Se usan los cuatro tipos de filtro del material: condiciones booleanas, `between()`, `isin()` y `query()`.

In [19]:
# Condición booleana + between(): parque de PASAJEROS a cierre de año (diciembre) en la ventana 2021–2025
pasajeros_dic = historial[
    historial["primary_use"].eq("Passenger")
    & historial["month"].eq(12)
    & historial["year"].between(2021, 2025)
]

# between(): vehículos del registro con año de modelo dentro de la ventana
registro_ventana = registro[registro["model_year"].between(2021, 2025)]

# query(): ventas y PM2.5 en los años con datos de ambos dentro de la ventana
ventas_ventana = ventas.query("2021 <= year <= 2023")
pm25_ventana = pm25.query("2021 <= year <= 2023")

# isin(): los 10 países con más ventas de eléctricos en 2023
top10_iso = ventas.query("year == 2023").nlargest(10, "ev_sales")["iso3"]
ventas_top10 = ventas[ventas["iso3"].isin(top10_iso)]

resumen_filtros = pd.DataFrame(
    {
        "filtro": ["boolean + between", "between", "query", "query", "isin"],
        "resultado": ["pasajeros_dic", "registro_ventana", "ventas_ventana", "pm25_ventana", "ventas_top10"],
        "filas": [len(pasajeros_dic), len(registro_ventana), len(ventas_ventana), len(pm25_ventana), len(ventas_top10)],
    }
)
assert pasajeros_dic["date"].dt.month.eq(12).all() and len(pasajeros_dic) == 39 * 5
resumen_filtros

,filtro,resultado,filas
0,boolean + between,pasajeros_dic,195
1,between,registro_ventana,197227
2,query,ventas_ventana,177
3,query,pm25_ventana,600
4,isin,ventas_top10,159


### 2.2 I1 · Participación de eléctricos en el parque de pasajeros de Washington
Se agrupa por año con `groupby().agg()` y se recalcula el porcentaje a partir de los totales (no se promedian
porcentajes de condados, porque cada condado tiene un tamaño distinto).

In [20]:
wa_anual = (
    historial[historial["primary_use"].eq("Passenger") & historial["month"].eq(12)]
    .groupby("year")
    .agg(bev=("bev", "sum"), phev=("phev", "sum"), ev_total=("ev_total", "sum"), total_vehicles=("total_vehicles", "sum"))
    .assign(
        pct_ev=lambda d: d["ev_total"] / d["total_vehicles"] * 100,
        pct_bev_of_ev=lambda d: d["bev"] / d["ev_total"] * 100,
        ev_growth_pct=lambda d: d["ev_total"].pct_change() * 100,
        pct_ev_change_pp=lambda d: d["pct_ev"].diff(),
    )
)
wa_anual.round(2)

,bev,phev,ev_total,total_vehicles,pct_ev,pct_bev_of_ev,ev_growth_pct,pct_ev_change_pp
year,,,,,,,,
2017,18666,9819,28485,4737013,0.60,65.53,NaN,NaN
2018,27007,13920,40927,4834781,0.85,65.99,43.68,0.25
2019,36849,16550,53399,4919364,1.09,69.01,30.47,0.24
2020,46262,18295,64557,4905880,1.32,71.66,20.90,0.23
2021,63161,23007,86168,4925098,1.75,73.30,33.48,0.43
2022,86941,27617,114558,4867159,2.35,75.89,32.95,0.60
2023,127459,36808,164267,4867472,3.37,77.59,43.39,1.02
2024,171533,46822,218355,4904615,4.45,78.56,32.93,1.08
2025,210956,55849,266805,4963432,5.38,79.07,22.19,0.92


In [21]:
v21, v25 = wa_anual.loc[2021], wa_anual.loc[2025]
tcac = ((v25["ev_total"] / v21["ev_total"]) ** (1 / 4) - 1) * 100
I1 = {
    "indicador": "I1 · % eléctricos en parque de pasajeros WA",
    "valor_inicial": round(v21["pct_ev"], 2),
    "valor_final": round(v25["pct_ev"], 2),
    "cambio": round(v25["pct_ev"] - v21["pct_ev"], 2),
    "unidad": "% (2021 → 2025, cambio en p. p.)",
}
md(f"""**I1.** En diciembre de 2021 los eléctricos eran el **{pct(v21['pct_ev'])}** de los vehículos de pasajeros de Washington
y en diciembre de 2025 el **{pct(v25['pct_ev'])}**: +{fmt(v25['pct_ev'] - v21['pct_ev'], 2)} puntos porcentuales.
El número de eléctricos pasó de {fmt(v21['ev_total'])} a {fmt(v25['ev_total'])} (×{fmt(v25['ev_total'] / v21['ev_total'], 1)};
crecimiento anual compuesto de {pct(tcac, 1)}), mientras el parque total casi no cambió
({fmt(v21['total_vehicles'])} → {fmt(v25['total_vehicles'])}). **La primera parte de la pregunta se confirma:** los
eléctricos aumentaron con fuerza en la ventana de estudio.""")

**I1.** En diciembre de 2021 los eléctricos eran el **1,75 %** de los vehículos de pasajeros de Washington
y en diciembre de 2025 el **5,38 %**: +3,63 puntos porcentuales.
El número de eléctricos pasó de 86.168 a 266.805 (×3,1;
crecimiento anual compuesto de 32,7 %), mientras el parque total casi no cambió
(4.925.098 → 4.963.432). **La primera parte de la pregunta se confirma:** los
eléctricos aumentaron con fuerza en la ventana de estudio.

### 2.3 Ordenamiento y tabla dinámica por condado
`pivot_table()` arma una tabla condado × año con el % de eléctricos. Se ordena con `sort_values()` para ver los condados
con mayor adopción en 2025 y se categoriza a los 39 condados en niveles de adopción con `pd.cut()`.

In [22]:
pivote_condados = (
    pasajeros_dic.pivot_table(index="county", columns="year", values=["ev_total", "total_vehicles"], aggfunc="sum", observed=True)
)
pct_condado = (pivote_condados["ev_total"] / pivote_condados["total_vehicles"] * 100).round(2)
pct_condado["cambio_pp"] = pct_condado[2025] - pct_condado[2021]
pct_condado = pct_condado.sort_values(2025, ascending=False)
pct_condado.head(10)

year,2021,2022,2023,2024,2025,cambio_pp
county,,,,,,
King,3.26,4.33,6.16,7.91,9.32,6.06
San Juan,4.25,5.00,6.07,7.30,8.43,4.18
Snohomish,1.70,2.39,3.60,4.92,6.09,4.39
Jefferson,2.28,2.85,3.74,4.64,5.56,3.28
Clark,1.55,2.17,3.04,4.06,4.99,3.44
Kitsap,1.69,2.18,3.04,3.99,4.84,3.15
Island,1.66,2.12,2.85,3.73,4.66,3.00
Thurston,1.42,1.92,2.68,3.59,4.38,2.96
Whatcom,1.52,1.94,2.66,3.49,4.22,2.70


In [23]:
NIVELES = ["Baja (<2 %)", "Media (2–5 %)", "Alta (≥5 %)"]
pct_condado["nivel_adopcion_2025"] = pd.cut(pct_condado[2025], bins=[0, 2, 5, 100], labels=NIVELES, right=False)
niveles = (
    pct_condado.groupby("nivel_adopcion_2025", observed=False)
    .agg(condados=(2025, "size"), pct_ev_promedio_2025=(2025, "mean"), cambio_pp_promedio=("cambio_pp", "mean"))
    .round(2)
)
niveles

,condados,pct_ev_promedio_2025,cambio_pp_promedio
nivel_adopcion_2025,,,
Baja (<2 %),15,1.27,0.89
Media (2–5 %),20,3.27,2.21
Alta (≥5 %),4,7.35,4.48


In [24]:
alto = pct_condado.index[0]
md(f"""**Lectura.** La adopción está concentrada: **{alto}** lidera con {pct(pct_condado.loc[alto, 2025])} de eléctricos en 2025,
y {niveles.loc[NIVELES[2], 'condados']} condados superan el 5 %, mientras {niveles.loc[NIVELES[0], 'condados']} siguen por debajo del 2 %.
Todos los condados subieron entre 2021 y 2025 (mínimo +{fmt(pct_condado['cambio_pp'].min(), 2)} p. p.). Para la pregunta analítica
esto importa porque el efecto sobre el aire, si existe, debería notarse primero en los condados urbanos de alta adopción.""")

**Lectura.** La adopción está concentrada: **King** lidera con 9,32 % de eléctricos en 2025,
y 4 condados superan el 5 %, mientras 15 siguen por debajo del 2 %.
Todos los condados subieron entre 2021 y 2025 (mínimo +0,17 p. p.). Para la pregunta analítica
esto importa porque el efecto sobre el aire, si existe, debería notarse primero en los condados urbanos de alta adopción.

### 2.4 I4 · Tipo de eléctrico en los modelos 2021–2025 (tabla dinámica año × tipo)

In [25]:
pivote_tipo = registro_ventana.pivot_table(index="model_year", columns="ev_type", values="dol_vehicle_id", aggfunc="count", observed=True)
pivote_tipo["pct_bev"] = (pivote_tipo["BEV"] / pivote_tipo[["BEV", "PHEV"]].sum(axis=1) * 100).round(2)

autonomia = (
    registro_ventana.groupby("ev_type", observed=True)
    .agg(vehiculos=("dol_vehicle_id", "count"),
         autonomia_media_millas=("electric_range", "mean"),
         autonomia_con_dato=("electric_range", "count"))
    .round(1)
)
display(pivote_tipo)
autonomia

ev_type,BEV,PHEV,pct_bev
model_year,,,
2021,16210,4394,78.67
2022,24840,5004,83.23
2023,53913,7403,87.93
2024,38373,10606,78.35
2025,29005,7479,79.50


,vehiculos,autonomia_media_millas,autonomia_con_dato
ev_type,,,
BEV,162341,217.6,793
PHEV,34886,31.6,34878


In [26]:
pct_bev_ventana = registro_ventana["ev_type"].eq("BEV").mean() * 100
I4 = {
    "indicador": "I4 · % BEV entre eléctricos modelo 2021–2025 (WA)",
    "valor_inicial": pivote_tipo.loc[2021, "pct_bev"],
    "valor_final": pivote_tipo.loc[2025, "pct_bev"],
    "cambio": round(pivote_tipo.loc[2025, "pct_bev"] - pivote_tipo.loc[2021, "pct_bev"], 2),
    "unidad": "% (modelo 2021 → 2025, cambio en p. p.)",
}
md(f"""**I4.** De los {fmt(len(registro_ventana))} eléctricos registrados con modelo 2021–2025, el **{pct(pct_bev_ventana)}** son de batería (BEV),
que no emiten por el tubo de escape. La participación BEV pasó de {pct(I4['valor_inicial'])} (modelo 2021) a {pct(I4['valor_final'])}
(modelo 2025). Es un matiz para la pregunta: los híbridos enchufables (PHEV) todavía queman gasolina, así que no todo el
crecimiento de eléctricos se traduce igual en menos emisiones. La autonomía media se calcula solo sobre los vehículos con
dato ({fmt(autonomia['autonomia_con_dato'].sum())} de {fmt(autonomia['vehiculos'].sum())}); con los ceros originales habría salido mucho más baja.""")

**I4.** De los 197.227 eléctricos registrados con modelo 2021–2025, el **82,31 %** son de batería (BEV),
que no emiten por el tubo de escape. La participación BEV pasó de 78,67 % (modelo 2021) a 79,50 %
(modelo 2025). Es un matiz para la pregunta: los híbridos enchufables (PHEV) todavía queman gasolina, así que no todo el
crecimiento de eléctricos se traduce igual en menos emisiones. La autonomía media se calcula solo sobre los vehículos con
dato (35.671 de 197.227); con los ceros originales habría salido mucho más baja.

### 2.5 I2 · Crecimiento de las ventas de eléctricos por país (2021 → 2023)

In [27]:
ventas_pivote = ventas_ventana.pivot_table(index=["iso3", "country"], columns="year", values="ev_sales", aggfunc="sum")
ventas_pivote = ventas_pivote.dropna(subset=[2021, 2023])          # solo países con dato en ambos años
ventas_pivote = ventas_pivote[ventas_pivote[2021] >= 1000]         # crecimiento estable: base mínima de 1.000 ventas
ventas_pivote["ev_sales_growth_pct"] = (ventas_pivote[2023] / ventas_pivote[2021] - 1) * 100
ventas_pivote = ventas_pivote.sort_values("ev_sales_growth_pct", ascending=False)

I2 = {
    "indicador": "I2 · Crecimiento mediano de ventas EV por país",
    "valor_inicial": None,
    "valor_final": round(ventas_pivote["ev_sales_growth_pct"].median(), 2),
    "cambio": None,
    "unidad": f"% (2021 → 2023, {len(ventas_pivote)} países)",
}
display(ventas_pivote.head(10).round(1))
md(f"""**I2.** En los {len(ventas_pivote)} países con al menos 1.000 ventas en 2021, la mediana de crecimiento de las ventas de
eléctricos entre 2021 y 2023 fue de **{pct(I2['valor_final'], 1)}**. {ventas_pivote['ev_sales_growth_pct'].gt(0).sum()} países crecieron
y {ventas_pivote['ev_sales_growth_pct'].le(0).sum()} no. Se exige una base mínima para que un país que pasa de 10 a 100 ventas
(+900 %) no distorsione la comparación.""")

,year,2021,2022,2023,ev_sales_growth_pct
iso3,country,,,,
TUR,Turkey,3920.0,7540.0,67500.0,1621.9
THA,Thailand,9000.0,20900.0,89000.0,888.9
MYS,Malaysia,1250.0,3200.0,11100.0,788.0
IND,India,12050.0,48023.0,82270.0,582.7
NPL,Nepal,1153.0,2596.0,7491.0,549.7
CRI,Costa Rica,1130.0,2661.0,6337.0,460.8
AUS,Australia,20400.0,38900.0,98000.0,380.4
BRA,Brazil,13900.0,18500.0,52000.0,274.1
ARE,United Arab Emirates,5100.0,13700.0,16900.0,231.4


**I2.** En los 47 países con al menos 1.000 ventas en 2021, la mediana de crecimiento de las ventas de
eléctricos entre 2021 y 2023 fue de **100,0 %**. 45 países crecieron
y 2 no. Se exige una base mínima para que un país que pasa de 10 a 100 ventas
(+900 %) no distorsione la comparación.

### 2.6 I3 · Variación del PM2.5 por país (2021 → 2023)

In [28]:
pm25_pivote = pm25_ventana.pivot_table(index=["iso3", "country", "region"], columns="year", values="pm25", observed=True).dropna()
pm25_pivote["pm25_change"] = pm25_pivote[2023] - pm25_pivote[2021]
pm25_pivote["pm25_change_pct"] = pm25_pivote["pm25_change"] / pm25_pivote[2021] * 100

por_region = (
    pm25_pivote.groupby(level="region", observed=True)
    .agg(paises=("pm25_change", "size"), pm25_2023_medio=(2023, "mean"),
         cambio_medio_ugm3=("pm25_change", "mean"), pct_paises_que_bajan=("pm25_change", lambda s: (s < 0).mean() * 100))
    .sort_values("cambio_medio_ugm3")
    .round(2)
)
por_region

,paises,pm25_2023_medio,cambio_medio_ugm3,pct_paises_que_bajan
region,,,,
South Asia,6,40.07,-1.64,66.67
North America,3,6.05,-1.11,100.00
Europe & Central Asia,52,14.27,-0.95,92.31
Latin America & Caribbean,35,16.21,-0.22,65.71
East Asia & Pacific,33,13.61,0.01,45.45
Sub-Saharan Africa,48,28.67,0.15,50.00
Middle East & North Africa,23,40.92,0.47,47.83


In [29]:
usa = pm25_pivote.xs("USA", level="iso3").iloc[0]
I3 = {
    "indicador": "I3 · Variación mediana del PM2.5 por país",
    "valor_inicial": round(pm25_pivote[2021].median(), 2),
    "valor_final": round(pm25_pivote[2023].median(), 2),
    "cambio": round(pm25_pivote["pm25_change"].median(), 2),
    "unidad": f"µg/m³ (2021 → 2023, {len(pm25_pivote)} países)",
}
md(f"""**I3.** Entre 2021 y 2023 el PM2.5 bajó en **{pct((pm25_pivote['pm25_change'] < 0).mean() * 100, 1)}** de los {len(pm25_pivote)} países
(cambio mediano {fmt(I3['cambio'], 2)} µg/m³). En **Estados Unidos**, el contexto nacional de Washington, pasó de {fmt(usa[2021], 2)} a
{fmt(usa[2023], 2)} µg/m³ ({pct(usa['pm25_change_pct'], 1)}). La baja es general y ocurre también en regiones con poca adopción de
eléctricos, lo que advierte que el PM2.5 depende de muchos factores (industria, incendios, clima), no solo del transporte.
La relación entre I2 e I3 se mide en la sección 3, después de unir los dos datasets.""")

**I3.** Entre 2021 y 2023 el PM2.5 bajó en **64,0 %** de los 200 países
(cambio mediano -0,26 µg/m³). En **Estados Unidos**, el contexto nacional de Washington, pasó de 8,23 a
6,97 µg/m³ (-15,3 %). La baja es general y ocurre también en regiones con poca adopción de
eléctricos, lo que advierte que el PM2.5 depende de muchos factores (industria, incendios, clima), no solo del transporte.
La relación entre I2 e I3 se mide en la sección 3, después de unir los dos datasets.

In [30]:
indicadores = pd.DataFrame([I1, I2, I3, I4])
indicadores

,indicador,valor_inicial,valor_final,cambio,unidad
0,I1 · % eléctricos en parque de pasajeros WA,1.75,5.38,3.63,"% (2021 → 2025, cambio en p. p.)"
1,I2 · Crecimiento mediano de ventas EV por país,NaN,100.00,NaN,"% (2021 → 2023, 47 países)"
2,I3 · Variación mediana del PM2.5 por país,17.42,17.32,-0.26,"µg/m³ (2021 → 2023, 200 países)"
3,I4 · % BEV entre eléctricos modelo 2021–2025 (WA),78.67,79.50,0.83,"% (modelo 2021 → 2025, cambio en p. p.)"
